In [16]:
import os
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Hi")
print(response.content)

Hello! How can I assist you today?


## Utils

In [18]:
## DocLoader
from langchain_community.document_loaders import PyMuPDFLoader
def resume_loader(file_path):
    loader = PyMuPDFLoader(file_path)
    doc_list =  loader.load()
    return '\n'.join([i.page_content for i in doc_list if i.page_content])

def text_loader(file_path):
    with open(file_path, 'r') as f:
        return f.read()

## Analyzer

In [19]:
### Resume Analyzer
resume_text = resume_loader("/Users/munna/Projects/QBS/careerpilot_demo/docs/Mahmud_Hasan_Munna_BL.pdf")
jd_text = text_loader("/Users/munna/Projects/QBS/careerpilot_demo/docs/jd.txt")

In [36]:
### Promt --> LLM --> Structured Output Response
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage
from pydantic import BaseModel, Field
from typing import List, Optional,Dict

In [21]:
def analyze_resume(resume_text, jd_text):
    class ResumeAnalysis(BaseModel):
        strong_points: List[str] = Field(description="List of strong points in the resume")
        weak_points: List[str] = Field(description="List of weak points in the resume")

    analyze_llm = llm.with_structured_output(ResumeAnalysis)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Resume Strong and Weakness Analysis.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured analysis highlighting the strong points and weak points of the resume with respect to the job description.
                - Max allowed strong points: 5
                - Max allowed weak points: 5

                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    analyze_chain = prompt | analyze_llm
    result = analyze_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [22]:
analyze_resume(resume_text, jd_text)

{'strong_points': ["4+ years of experience in AI/ML engineering with a focus on deploying production-grade machine learning systems, aligning with the job's requirement for extensive experience in AI/ML engineering.",
  "Hands-on expertise in LLM/GenAI systems, including RAG architectures and prompt engineering, which directly relates to the job's focus on Generative AI platforms and applications.",
  'Experience with MLOps, CI/CD for ML, and cloud platforms (AWS), which is essential for architecting scalable and cost-efficient AI infrastructure as mentioned in the job description.',
  'Proven track record in automating processes and improving efficiency, such as reducing onboarding time by 90% and saving 80 hours/month, demonstrating the ability to drive impactful AI solutions.',
  'Strong technical skills in Python, FastAPI, Docker, and various ML frameworks, which are crucial for backend engineering and operationalizing production-scale GenAI applications.'],
 'weak_points': ['Only 

## Resume Feedback

In [23]:
def get_resume_feedback(resume_text, jd_text):
    class ResumeFeedback(BaseModel):
        to_be_added: List[str] = Field(description="Elements that are missing in the resume but are relevant to the job description")
        to_be_deleted: List[str] = Field(description="Elements that are present in the resume but are not relevant to the job description")

    feedback_llm = llm.with_structured_output(ResumeFeedback)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Resume Providing Resume Feedback.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured feedback highlighting the elements that are missing in the resume but are relevant to the job description (to_be_added) and the elements that are present in the resume but are not relevant to the job description (to_be_deleted).


                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    feedback_chain = prompt | feedback_llm
    result = feedback_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [24]:
get_resume_feedback(resume_text, jd_text)

{'to_be_added': ['Experience leading technical teams or AI engineering squads',
  'Expertise in architecting distributed systems and operationalizing production-scale GenAI applications',
  'Hands-on experience with LLM application development, prompt engineering frameworks, RAG architectures, AI agents and tool-calling systems',
  'Experience with AI infrastructure including GPU-based inference systems, open-source LLM deployment, fine-tuning techniques (LoRA/PEFT)',
  'Experience with AI governance, responsible AI, and enterprise compliance frameworks',
  'Strong stakeholder influence, business communication, and presentation skills',
  'Contributions to open-source projects, patents, conferences, or published research'],
 'to_be_deleted': ['Experience in telecom-domain processing 1B+ records (not relevant to GenAI)',
  'Specific mention of telecom recharge forecasting (not relevant to the broader GenAI focus)',
  'Details on financial and operational reports (not directly related to

### Study Plan

In [25]:
class SkillGapAnalysis(BaseModel):
    skill: List[str] = Field(description="List of skills that are required for the job but are missing in the resume")


In [32]:
def get_skill_gap(resume_text, jd_text):
    class SkillGapAnalysis(BaseModel):
        skill: List[str] = Field(description="List of skills that are required for the job but are missing in the resume")

    skill_gap_llm = llm.with_structured_output(SkillGapAnalysis)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are an expert AI assistant specializing in Skill Gap Analysis.

                Responsibilities:
                - You will be provided with a resume and a job description.
                - Analyze the resume in the context of the job description.
                - You will provide a structured output listing the technical skills that are required for the job but are missing in the resume.
                - Only list the technical skills(theory and hands-on) that are relevant to the job description and are missing in the resume. Do not list any other elements.

                """
            ),
            (
                "human",
                """
                Resume:
                {resume}

                JD:
                {jd}
                """
            ),
        ]
    )
    skill_gap_chain = prompt | skill_gap_llm
    result = skill_gap_chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
        }
    )
    return dict(result)

In [46]:
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


def study_plan_based_on_skill_gaps(
    resume_text: str,
    jd_text: str,
    days_for_study: int = 30
):
    """
    Generate a personalized day-by-day study plan based on skill gaps.
    """

    # -----------------------------
    # Structured Output Models
    # -----------------------------
    class DayPlan(BaseModel):
        topic: str = Field(
            description="Main topic to study on this day"
        )
        subtopics: List[str] = Field(
            description="Detailed subtopics to cover"
        )

    class StudyPlan(BaseModel):
        study_plan: List[DayPlan] = Field(
            description="""
            Dictionary where keys are day_1, day_2, day_3 ... day_N.
            N must equal the number of study days provided.
            """
        )

    # -----------------------------
    # Get Skill Gaps
    # -----------------------------
    skills = get_skill_gap(
        resume_text,
        jd_text
    )["skill"]

    # -----------------------------
    # Structured Output LLM
    # -----------------------------
    study_plan_llm = llm.with_structured_output(StudyPlan)

    # -----------------------------
    # Prompt
    # -----------------------------
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
You are an expert AI career coach and technical mentor.

Your task is to create a detailed study plan for a candidate.

Rules:

1. Analyze the missing skills carefully.
2. Create a study plan spanning EXACTLY the number of days provided.
3. The output keys MUST be:
   day_1, day_2, day_3 ... day_N
4. N must equal the study days provided.
5. Each day must contain:
   - topic
   - subtopics
6. Progress from beginner to advanced.
7. Prioritize high-impact skills first.
8. Ensure all missing skills are covered.
9. If days are more than required, distribute learning, revision,
   hands-on projects, interview preparation, and mock assessments.
10. Keep topics practical and job-focused.
11. Each day should contain 3-6 subtopics.
12. Return ONLY the structured output.
                """
            ),
            (
                "human",
                """
Resume:
{resume}

Job Description:
{jd}

Missing Skills:
{skills}

Study Days:
{days}

Create a day-by-day study plan.
                """
            ),
        ]
    )

    # -----------------------------
    # Chain
    # -----------------------------
    chain = prompt | study_plan_llm

    result = chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
            "skills": skills,
            "days": days_for_study,
        }
    )

    return result.model_dump()


# Example Usage
study_plan = study_plan_based_on_skill_gaps(
    resume_text=resume_text,
    jd_text=jd_text,
    days_for_study=7
)



In [47]:
study_plan

{'study_plan': [{'topic': 'Introduction to LLMOps',
   'subtopics': ['Understanding LLMOps and its importance',
    'Key components of LLMOps',
    'Best practices for LLMOps implementation']},
  {'topic': 'RAG Pipelines and Agent Orchestration',
   'subtopics': ['Overview of RAG architectures',
    'Building RAG pipelines',
    'Agent orchestration techniques and tools']},
  {'topic': 'Prompt Management and AI Governance',
   'subtopics': ['Effective prompt engineering strategies',
    'AI governance frameworks',
    'Establishing AI ethics and compliance standards']},
  {'topic': 'Model Evaluation and AI Observability',
   'subtopics': ['Techniques for model evaluation',
    'Monitoring AI systems for performance',
    'Implementing AI observability best practices']},
  {'topic': 'Performance Optimization Techniques',
   'subtopics': ['Optimizing for latency and throughput',
    'GPU utilization strategies',
    'Managing inference costs effectively']},
  {'topic': 'Fine-tuning Techn

## Interview QNA

In [44]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class InterviewQnA(BaseModel):
    question: str = Field(
        description="Interview question"
    )

    options: List[str] = Field(
        description="Exactly 4 answer options"
    )

    answer: str = Field(
        description="Correct answer"
    )

    explanation: str = Field(
        description="Explanation of why the answer is correct"
    )


class InterviewPreparation(BaseModel):
    qna: List[InterviewQnA] = Field(
        description="List of interview questions and answers"
    )


def generate_interview_qna(
    resume_text: str,
    jd_text: str,
    num_questions: int = 10
):

    interview_llm = llm.with_structured_output(
        InterviewPreparation
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
You are an expert technical interviewer.

Responsibilities:

1. Analyze the candidate's resume.
2. Analyze the job description.
3. Generate interview questions relevant to:
   - Required skills
   - Candidate experience
   - Missing skills
   - Real-world job responsibilities

Rules:

1. Generate EXACTLY the requested number of questions.
2. Each question must have EXACTLY 4 options.
3. Only ONE option should be correct.
4. Include the correct answer.
5. Include a short explanation.
6. Mix difficulty levels:
   - 30% Easy
   - 50% Medium
   - 20% Hard
7. Focus on practical interview questions.
8. Avoid duplicate questions.
                """
            ),
            (
                "human",
                """
Resume:
{resume}

Job Description:
{jd}

Generate exactly {num_questions} interview questions.
                """
            )
        ]
    )

    chain = prompt | interview_llm

    result = chain.invoke(
        {
            "resume": resume_text,
            "jd": jd_text,
            "num_questions": num_questions
        }
    )

    return result.model_dump()

In [45]:
generate_interview_qna(resume_text, jd_text)

{'qna': [{'question': 'What is the primary purpose of RAG in Generative AI systems?',
   'options': ['To manage user authentication',
    'To enhance model training',
    'To retrieve relevant information from large datasets',
    'To optimize database queries'],
   'answer': 'To retrieve relevant information from large datasets',
   'explanation': "RAG (Retrieval-Augmented Generation) is used in Generative AI to enhance the model's ability to generate responses by retrieving relevant information from large datasets."},
  {'question': 'Which of the following is a common framework used for building conversational AI systems?',
   'options': ['TensorFlow', 'FastAPI', 'LangChain', 'Dask'],
   'answer': 'LangChain',
   'explanation': 'LangChain is specifically designed for building applications that involve language models, making it a common choice for conversational AI systems.'},
  {'question': "In the context of AI/ML, what does the term 'hallucination' refer to?",
   'options': ['The 